# 02 — Drift investigation
Compare every weekly feature distribution against the fixed reference dataset.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from src.core import DATA, drift_report, prepare_xy, load_production_model, metrics_for
ref = pd.read_csv(DATA / 'churn_reference_train.csv')
all_drift=[]
all_perf=[]
model,entry=load_production_model()
for w in range(1,9):
    week=pd.read_csv(DATA / f'churn_week_{w}.csv')
    d=drift_report(ref,week); d['week']=f'Week {w}'; all_drift.append(d)
    X,y=prepare_xy(week); m=metrics_for(model,X,y); m['week']=f'Week {w}'; all_perf.append(m)
drift=pd.concat(all_drift,ignore_index=True)
performance=pd.DataFrame(all_perf)
drift.head(), performance

In [ ]:
heat=drift.pivot(index='feature',columns='week',values='psi')
plt.figure(figsize=(12, max(4, len(heat)*0.35)))
plt.imshow(heat, aspect='auto', interpolation='nearest')
plt.yticks(range(len(heat.index)), heat.index)
plt.xticks(range(len(heat.columns)), heat.columns, rotation=45)
plt.colorbar(label='PSI')
plt.title('Feature PSI by week')
plt.tight_layout()
plt.show()
performance.plot(x='week', y=['f1','roc_auc'], marker='o', title='Weekly performance')
plt.xticks(rotation=45); plt.tight_layout(); plt.show()